# 03 · Placar das três configurações

Cobre os itens **06** (placar das três configurações com resultados interpretados) e
**07** (5 erros reais com proposta de correção) da pontuação.

> ⚠️ Requer a LLM local no ar. A avaliação completa (27 casos × 3 configurações)
> leva alguns minutos na RX 6600.

In [1]:
import sys, json
sys.path.append("..")
from src.avaliar import main

main()  # salva avaliacao/resultados.json e avaliacao/placar.md


=== Avaliando configuração: sem_contexto ===
[sem_contexto] q01 ERR (2.5s)
[sem_contexto] q02 OK  (3.2s)
[sem_contexto] q03 ERR (2.5s)
[sem_contexto] q04 ERR (2.5s)
[sem_contexto] q05 ERR (2.5s)
[sem_contexto] q06 ERR (3.0s)
[sem_contexto] q07 ERR (2.5s)
[sem_contexto] q08 ERR (2.5s)
[sem_contexto] q09 ERR (2.5s)
[sem_contexto] q10 ERR (2.6s)
[sem_contexto] q11 ERR (2.5s)
[sem_contexto] q12 ERR (2.5s)
[sem_contexto] q13 OK  (2.7s)
[sem_contexto] q14 ERR (2.5s)
[sem_contexto] q15 OK  (2.8s)
[sem_contexto] q16 ERR (2.5s)
[sem_contexto] q17 ERR (2.9s)
[sem_contexto] q18 ERR (2.5s)
[sem_contexto] q19 ERR (3.2s)
[sem_contexto] q20 ERR (2.6s)
[sem_contexto] q21 ERR (3.0s)
[sem_contexto] q22 ERR (2.6s)
[sem_contexto] q23 OK  (2.5s)
[sem_contexto] q24 OK  (2.5s)
[sem_contexto] q25 OK  (2.5s)
[sem_contexto] q26 OK  (2.5s)
[sem_contexto] q27 ERR (2.6s)

=== Avaliando configuração: rag ===
[rag] q01 ERR (2.7s)
[rag] q02 OK  (4.4s)
[rag] q03 OK  (2.9s)
[rag] q04 OK  (3.4s)
[rag] q05 ERR (2.9s)
[r

## Placar

In [2]:
import pandas as pd
resultados = json.loads(open("../avaliacao/resultados.json", encoding="utf-8").read())
placar = pd.DataFrame([r["metricas"] for r in resultados.values()])
placar

,config,acuracia,alucinacao,json_valido,citacao_correta,tempo_medio_s
0,sem_contexto,0.259,0.2,1.0,0.000,2.6
1,rag,0.741,0.0,1.0,0.842,2.9
2,agente,0.704,0.0,1.0,1.000,4.8


### Interpretação

| Configuração | Acurácia | Alucinação | Citação correta | Tempo |
|---|---|---|---|---|
| sem_contexto | 26% | 20% | 0% | 2,6 s |
| rag | **74%** | **0%** | 84% | 2,9 s |
| agente | 70% | **0%** | **100%** | 4,8 s |

- **sem_contexto**: o modelo de 3B não conhece as regras de São Paulo. Recusa a maioria das perguntas e, quando arrisca, inventa fontes (`"123"`, `"tapa-buraco.sp"`). Alucinou em 1 de 5 casos "não consta".
- **rag**: a acurácia quase triplica e a alucinação cai a zero (limiar da busca + regra de recusa no prompt).
- **agente**: mantém 0% de alucinação e leva a citação correta a 100%. Em troca, recusa um pouco mais (70% vs 74%) e demora mais (mais chamadas à LLM).

**Conclusão: a IA com contexto acerta 2,8x mais e não alucina, enquanto o mesmo modelo sem contexto alucina.** O agente é a configuração escolhida para o MVP, porque sempre cita a fonte certa.

## Os 5 erros reais

A célula abaixo lista os casos errados de cada configuração. Escolha 5, descreva a causa
e a correção proposta no relatório (`RELATORIO.md`, seção 6).

In [3]:
for nome, r in resultados.items():
    erros = [e for e in r["execucoes"] if not e["correto"]]
    print(f"\n===== {nome}: {len(erros)} erros =====")
    for e in erros:
        print(f"- {e['caso_id']} ({e['tipo']}): {json.dumps(e['json'], ensure_ascii=False)[:160]}")
        print(f"    passos: {e['passos'][-1] if e['passos'] else '-'}")


===== sem_contexto: 20 erros =====
- q01 (respondivel): {"resposta": "não consta", "fonte_id": null, "confianca": 0.0}
    passos: pergunta direta ao modelo, sem contexto
- q03 (respondivel): {"resposta": "não consta", "fonte_id": null, "confianca": 0.0}
    passos: pergunta direta ao modelo, sem contexto
- q04 (respondivel): {"resposta": "Tapa Buraco", "fonte_id": 1, "confianca": 0.95}
    passos: pergunta direta ao modelo, sem contexto
- q05 (respondivel): {"resposta": "não consta", "fonte_id": null, "confianca": 0.0}
    passos: pergunta direta ao modelo, sem contexto
- q06 (respondivel): {"resposta": "Imagens do local do acidente, descrição detalhada do ocorrido, número de telefone de testemunhas, e detalhes do seu equipamento de segurança", "fo
    passos: pergunta direta ao modelo, sem contexto
- q07 (respondivel): {"resposta": "não consta", "fonte_id": null, "confianca": 0.0}
    passos: pergunta direta ao modelo, sem contexto
- q08 (respondivel): {"resposta": "não consta", "fo